# Notebook 04 — Building Realistic Weather Profiles

This is the capstone notebook of the series. We take the cleaned historical
weather data from Notebook 03 and build **statistical profiles** that describe
what realistic weather looks like for each city in each month.

## The Problem

Our app's minions currently generate random forecasts:
- Temperature: uniform random between -20°C and 55°C
- Summary: randomly chosen from 10 labels, regardless of temperature

This produces implausible data like "Scorching" at -10°C or 50°C days in London.

## The Solution

A **weather profile** captures the statistical properties of real weather:
- What's the average temperature in New York in January? (around 0°C)
- How much does it vary? (standard deviation ~5°C)
- What fraction of January days are "Freezing" vs "Chilly" vs "Cool"?

With these profiles, minions can **sample** from realistic distributions
instead of uniform random ranges.

## What you will learn

1. Computing monthly statistics with `groupby()` and `.agg()`
2. Building probability distributions for categorical labels
3. Laplace smoothing (preventing zero probabilities)
4. Visualizing profiles with heatmaps and stacked bar charts
5. Sampling from a profile and validating it against real data
6. Exporting the profile as JSON for the app to consume

## Pre-requisites

- Notebook 03 must have been run (it saves cleaned data to MinIO)

---

In [ ]:
# Cell 1 — Imports
import sys
import os
import json
import io

SHARED_PATH = '/home/jovyan/work/shared'
if SHARED_PATH not in sys.path:
    sys.path.insert(0, SHARED_PATH)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from minio_helper import get_client, read_parquet, object_exists, ensure_bucket

%matplotlib inline
sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.dpi'] = 120

# Our app's 10 summary labels, ordered from coldest to hottest
SUMMARY_LABELS = [
    'Freezing', 'Bracing', 'Chilly', 'Cool', 'Mild',
    'Warm', 'Balmy', 'Hot', 'Sweltering', 'Scorching',
]

MONTH_NAMES = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']

client = get_client()
print('Imports OK')

In [ ]:
# Cell 2 — Load cleaned data from MinIO
# Notebook 03 saved these Parquet files to the clean-weather bucket.

CLEAN_BUCKET = 'clean-weather'

frames = []

# Load Open-Meteo cleaned data (preferred — already in °C, consistent columns)
meteo_obj = 'open_meteo_daily_cleaned.parquet'
if object_exists(client, CLEAN_BUCKET, meteo_obj):
    meteo_df = read_parquet(client, CLEAN_BUCKET, meteo_obj)
    # Standardize column names
    col_map = {
        'temperature_2m_mean': 'temp_mean_c',
        'temperature_2m_max': 'temp_max_c',
        'temperature_2m_min': 'temp_min_c',
        'precipitation_sum': 'precipitation_mm',
    }
    meteo_df = meteo_df.rename(columns=col_map)
    frames.append(meteo_df)
    print(f'Loaded Open-Meteo: {len(meteo_df):,} rows')

# Load GHCN cleaned data
ghcn_obj = 'ghcn_daily_cleaned.parquet'
if object_exists(client, CLEAN_BUCKET, ghcn_obj):
    ghcn_df = read_parquet(client, CLEAN_BUCKET, ghcn_obj)
    frames.append(ghcn_df)
    print(f'Loaded GHCN: {len(ghcn_df):,} rows')

if not frames:
    raise FileNotFoundError(
        'No cleaned data found in MinIO. Run Notebook 03 first.'
    )

# Combine both sources
# Use only columns that exist in both DataFrames
common_cols = ['date', 'location', 'source', 'temp_mean_c', 'summary']
for f in frames:
    # Keep only columns that exist
    missing = [c for c in common_cols if c not in f.columns]
    if missing:
        print(f'  Warning: missing columns {missing} in one source')

df = pd.concat(
    [f[[c for c in common_cols if c in f.columns]] for f in frames],
    ignore_index=True,
)

df['date'] = pd.to_datetime(df['date'])
df['month'] = df['date'].dt.month

print(f'\nCombined dataset: {len(df):,} rows')
print(f'Locations: {sorted(df["location"].unique())}')
df.head()

## Part 1: Monthly Temperature Statistics

For each location and month, we compute:
- **Mean** — the expected temperature
- **Standard deviation** — how much it varies day to day
- **Min/Max** — the observed extremes

These four numbers define a temperature distribution that a minion
can sample from using a **truncated normal distribution**:
```
temperature = clip(normal(mean, std), min, max)
```

In [ ]:
# Cell 4 — Compute monthly temperature statistics
# groupby(['location', 'month']) splits the data into subgroups.
# .agg() computes multiple aggregation functions on each subgroup.

temp_stats = (
    df.groupby(['location', 'month'])['temp_mean_c']
    .agg(['mean', 'std', 'min', 'max', 'count'])
    .round(2)
)

# Rename for clarity
temp_stats.columns = ['temp_mean', 'temp_std', 'temp_min', 'temp_max', 'day_count']

print('Monthly temperature statistics (sample: New York):')
if 'new_york' in temp_stats.index.get_level_values('location'):
    display(temp_stats.loc['new_york'])
else:
    display(temp_stats.head(12))

In [ ]:
# Cell 5 — Heatmap: mean temperature by month and location
# This visualization shows the seasonal pattern for each city at a glance.

# Pivot for the heatmap: rows=location, columns=month
heatmap_data = temp_stats['temp_mean'].unstack(level='month')
heatmap_data.columns = MONTH_NAMES

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt='.1f',
    cmap='RdYlBu_r',  # Red=hot, Blue=cold
    center=15,         # Center the color scale at 15°C
    ax=ax,
    linewidths=0.5,
)
ax.set_title('Monthly Mean Temperature by Location (°C)', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('Location')
plt.tight_layout()
plt.show()

print('Key observations:')
print('  - Melbourne\'s seasons are reversed (Southern Hemisphere)')
print('  - Singapore is nearly constant year-round (equatorial)')
print('  - New York has the widest seasonal swing (~30°C range)')

In [ ]:
# Cell 6 — Violin plots: monthly temperature distributions
# Violins show the full shape of the distribution, not just summary stats.
# The wider the violin at a temperature, the more common that temperature is.

# Pick two contrasting cities
cities_to_plot = ['new_york', 'singapore']
available_cities = [c for c in cities_to_plot if c in df['location'].unique()]

if available_cities:
    subset = df[df['location'].isin(available_cities)]

    fig, axes = plt.subplots(1, len(available_cities), figsize=(7 * len(available_cities), 5))
    if len(available_cities) == 1:
        axes = [axes]

    for ax, city in zip(axes, available_cities):
        city_data = subset[subset['location'] == city]
        sns.violinplot(
            data=city_data,
            x='month',
            y='temp_mean_c',
            hue='month',
            palette='coolwarm',
            legend=False,
            ax=ax,
        )
        ax.set_xticklabels(MONTH_NAMES)
        ax.set_title(f'{city.replace("_", " ").title()} — Monthly Temperature', fontsize=12)
        ax.set_xlabel('Month')
        ax.set_ylabel('Daily Mean Temp (°C)')

    plt.tight_layout()
    plt.show()

## Part 2: Summary Label Probability Distributions

For each location and month, we compute the **probability** of each
Summary label. This tells us: "In New York in January, what fraction
of days are Freezing? Bracing? Chilly?"

### Laplace Smoothing

Some labels might have zero occurrences in our data. For example,
"Scorching" never happens in London. But we don't want a probability
of exactly 0.0 — that would mean a minion could *never* generate
that label, which is too rigid.

**Laplace smoothing** adds a small count (typically 1) to every label
before computing probabilities. This ensures every label has a tiny
but non-zero probability, while labels that actually occur remain
dominant.

In [ ]:
# Cell 8 — Compute label probabilities with Laplace smoothing

LAPLACE_ALPHA = 1  # Add this many "phantom" observations per label

def compute_label_probs(group, labels=SUMMARY_LABELS, alpha=LAPLACE_ALPHA):
    """
    Given a group of rows (one location+month), compute the probability
    of each Summary label with Laplace smoothing.

    Parameters
    ----------
    group : pd.DataFrame
        Subset of rows for one location+month.
    labels : list
        All possible label values.
    alpha : float
        Laplace smoothing parameter. 1 = add-one smoothing.

    Returns
    -------
    dict
        {label: probability} where probabilities sum to 1.0.
    """
    # Count actual occurrences
    counts = group['summary'].value_counts()

    # Apply Laplace smoothing: add alpha to each label
    smoothed = {label: counts.get(label, 0) + alpha for label in labels}

    # Normalize to probabilities
    total = sum(smoothed.values())
    probs = {label: round(count / total, 4) for label, count in smoothed.items()}

    return probs

# Compute for all location+month combinations
label_probs = {}
for (location, month), group in df.groupby(['location', 'month']):
    label_probs[(location, month)] = compute_label_probs(group)

# Show example: New York, January
example_key = None
for key in label_probs:
    if 'new_york' in key[0] and key[1] == 1:
        example_key = key
        break

if example_key:
    print(f'Label probabilities for {example_key[0]}, month {example_key[1]} (January):')
    for label, prob in label_probs[example_key].items():
        bar = '#' * int(prob * 100)
        print(f'  {label:12s}  {prob:.4f}  {bar}')
else:
    first_key = next(iter(label_probs))
    print(f'Label probabilities for {first_key}:')
    for label, prob in label_probs[first_key].items():
        print(f'  {label:12s}  {prob:.4f}')

In [ ]:
# Cell 9 — Stacked bar chart: label distribution by month for one city
# This shows how the mix of weather labels changes across seasons.

target_city = 'new_york'
if target_city not in df['location'].unique():
    target_city = df['location'].unique()[0]

# Build a DataFrame for plotting
stacked_data = pd.DataFrame(
    {month: label_probs.get((target_city, month), {}) for month in range(1, 13)}
).T
stacked_data.index = MONTH_NAMES

# Use our label order and a cold-to-hot colormap
stacked_data = stacked_data[SUMMARY_LABELS]
colors = plt.cm.coolwarm(np.linspace(0, 1, len(SUMMARY_LABELS)))

fig, ax = plt.subplots(figsize=(12, 6))
stacked_data.plot(
    kind='bar',
    stacked=True,
    color=colors,
    ax=ax,
    width=0.85,
)

ax.set_title(f'{target_city.replace("_", " ").title()} — Summary Label Distribution by Month', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('Probability')
ax.set_ylim(0, 1.0)
ax.legend(
    title='Summary',
    bbox_to_anchor=(1.01, 1),
    loc='upper left',
    fontsize=9,
)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print('Notice how the bar composition shifts from cold labels (blue) in winter')
print('to warm labels (red) in summer. This is exactly what minions should replicate.')

## Part 3: Build the Profile JSON

We combine the temperature statistics and label probabilities into a
single JSON structure that the app can load.

### Profile Structure

```json
{
  "new_york": {
    "1": {
      "temp_mean": 0.5,
      "temp_std": 4.8,
      "temp_min": -15.2,
      "temp_max": 12.1,
      "labels": {
        "Freezing": 0.25,
        "Bracing": 0.30,
        "Chilly": 0.20,
        ...
      }
    },
    "2": { ... },
    ...
  },
  "london": { ... }
}
```

### How a minion would use this

```python
# Pseudocode for MinionSchedulerService
profile = load_profile("new_york")
month = current_month()  # e.g., 1 for January
stats = profile[str(month)]

# Sample temperature from a normal distribution, clipped to observed range
temp = clip(normal(stats.temp_mean, stats.temp_std), stats.temp_min, stats.temp_max)

# Sample summary label according to learned probabilities
label = weighted_random(stats.labels)
```

In [ ]:
# Cell 11 — Build the profile dictionary

profile = {}

locations = sorted(df['location'].unique())

for location in locations:
    profile[location] = {}

    for month in range(1, 13):
        # Temperature statistics
        if (location, month) in temp_stats.index:
            stats = temp_stats.loc[(location, month)]
            temp_info = {
                'temp_mean': float(stats['temp_mean']),
                'temp_std': float(stats['temp_std']),
                'temp_min': float(stats['temp_min']),
                'temp_max': float(stats['temp_max']),
                'day_count': int(stats['day_count']),
            }
        else:
            temp_info = {
                'temp_mean': 15.0,
                'temp_std': 5.0,
                'temp_min': 0.0,
                'temp_max': 30.0,
                'day_count': 0,
            }

        # Label probabilities
        labels = label_probs.get((location, month), {})
        if not labels:
            # Uniform fallback if no data
            labels = {l: round(1.0 / len(SUMMARY_LABELS), 4) for l in SUMMARY_LABELS}

        profile[location][str(month)] = {
            **temp_info,
            'labels': labels,
        }

# Preview
print(f'Profile contains {len(profile)} locations, 12 months each.')
print(f'\nSample entry (new_york, January):')
sample = profile.get('new_york', profile.get(locations[0], {}))
print(json.dumps(sample.get('1', {}), indent=2))

In [ ]:
# Cell 12 — Validate: sample from the profile and compare to real data
# We generate 1000 synthetic forecasts using the profile and overlay
# the distribution on top of the real data to check if they match.

np.random.seed(42)  # Reproducibility

test_city = target_city
test_month = 7  # July
n_samples = 1000

p = profile[test_city][str(test_month)]

# Sample temperatures: normal distribution, clipped to min/max
sampled_temps = np.random.normal(p['temp_mean'], p['temp_std'], n_samples)
sampled_temps = np.clip(sampled_temps, p['temp_min'], p['temp_max'])

# Get real data for comparison
real_data = df[
    (df['location'] == test_city) &
    (df['month'] == test_month)
]['temp_mean_c'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: overlay histogram
axes[0].hist(real_data, bins=30, alpha=0.6, density=True, label='Real data', color='steelblue')
axes[0].hist(sampled_temps, bins=30, alpha=0.5, density=True, label='Profile samples', color='coral')
axes[0].set_title(f'{test_city.replace("_", " ").title()} — July Temperature', fontsize=12)
axes[0].set_xlabel('Temperature (°C)')
axes[0].set_ylabel('Density')
axes[0].legend()

# Right: label distribution comparison
# Sample labels according to profile probabilities
label_names = list(p['labels'].keys())
label_weights = list(p['labels'].values())
sampled_labels = np.random.choice(label_names, size=n_samples, p=label_weights)

# Count sampled labels
sampled_counts = pd.Series(sampled_labels).value_counts(normalize=True)

# Count real labels
real_labels = df[
    (df['location'] == test_city) &
    (df['month'] == test_month)
]['summary'].value_counts(normalize=True)

# Combine for side-by-side bar chart
comparison = pd.DataFrame({
    'Real': [real_labels.get(l, 0) for l in SUMMARY_LABELS],
    'Profile': [sampled_counts.get(l, 0) for l in SUMMARY_LABELS],
}, index=SUMMARY_LABELS)

comparison.plot(kind='bar', ax=axes[1], width=0.7)
axes[1].set_title(f'{test_city.replace("_", " ").title()} — July Labels: Real vs Profile', fontsize=12)
axes[1].set_xlabel('Summary Label')
axes[1].set_ylabel('Proportion')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Real data mean: {real_data.mean():.1f}°C, std: {real_data.std():.1f}°C')
print(f'Profile mean:   {p["temp_mean"]:.1f}°C, std: {p["temp_std"]:.1f}°C')
print(f'Sampled mean:   {sampled_temps.mean():.1f}°C, std: {sampled_temps.std():.1f}°C')
print('\nThe profile samples should closely match the real data distribution.')

In [ ]:
# Cell 13 — Radar/polar chart: label distribution by season
# This visualization shows how the mix of labels rotates through the year.

fig, axes = plt.subplots(1, 2, figsize=(14, 6), subplot_kw={'projection': 'polar'})

cities_for_radar = [test_city]
# Add a second contrasting city if available
for candidate in ['singapore', 'london', 'melbourne']:
    if candidate in profile and candidate != test_city:
        cities_for_radar.append(candidate)
        break

for ax, city in zip(axes, cities_for_radar):
    # For each month, get the dominant label
    angles = np.linspace(0, 2 * np.pi, 12, endpoint=False)

    # Plot mean temperature as radius
    temps = [profile[city][str(m)]['temp_mean'] for m in range(1, 13)]
    # Close the polygon
    temps_closed = temps + [temps[0]]
    angles_closed = np.append(angles, angles[0])

    ax.plot(angles_closed, temps_closed, 'o-', linewidth=2)
    ax.fill(angles_closed, temps_closed, alpha=0.2)

    ax.set_xticks(angles)
    ax.set_xticklabels(MONTH_NAMES)
    ax.set_title(f'{city.replace("_", " ").title()}\nMonthly Mean Temp (°C)', fontsize=11, pad=20)
    ax.set_ylim(bottom=min(0, min(temps) - 5))

plt.tight_layout()
plt.show()

In [ ]:
# Cell 14 — Upload the profile to MinIO
# This is the final artifact that the app can consume.

ANALYTICS_BUCKET = 'weather-analytics'
PROFILE_OBJECT = 'profiles/weather_profiles_v1.json'

ensure_bucket(client, ANALYTICS_BUCKET)

# Serialize to JSON
profile_json = json.dumps(profile, indent=2)

# Upload to MinIO
buf = io.BytesIO(profile_json.encode('utf-8'))
client.put_object(
    ANALYTICS_BUCKET,
    PROFILE_OBJECT,
    buf,
    length=len(profile_json.encode('utf-8')),
    content_type='application/json',
)

print(f'Profile uploaded to MinIO: {ANALYTICS_BUCKET}/{PROFILE_OBJECT}')
print(f'Size: {len(profile_json):,} bytes')
print(f'Locations: {list(profile.keys())}')
print(f'Months per location: 12')
print(f'Labels per month: {len(SUMMARY_LABELS)}')

## How to Use This Profile in the App

The weather profile JSON is now stored in MinIO at:
```
weather-analytics/profiles/weather_profiles_v1.json
```

### Integration with MinionSchedulerService

Currently, `MinionSchedulerService.cs` generates forecasts like this:

```csharp
// Current: purely random
var tempC = Random.Shared.Next(-20, 55);
var summary = summaries[Random.Shared.Next(summaries.Length)];
```

With the profile, it could be changed to:

```csharp
// Proposed: profile-driven
var month = DateTime.UtcNow.Month;
var stats = profile["new_york"][month.ToString()];

// Sample from normal distribution, clipped to observed range
var tempC = (int)Math.Round(
    Math.Clamp(
        NormalRandom(stats.temp_mean, stats.temp_std),
        stats.temp_min,
        stats.temp_max
    )
);

// Weighted random selection from label probabilities
var summary = WeightedRandom(stats.labels);
```

This produces forecasts that:
- Cluster around realistic temperatures for the current month
- Use summary labels that match the temperature
- Still have natural variability (from the normal distribution)
- Never produce impossible combinations (like "Freezing" at 40°C)

### Re-running the Profile

As more data is collected or new stations are added, simply re-run
Notebooks 03 and 04 to generate an updated `weather_profiles_v2.json`.
The Airflow quality report DAG (DAG 3) can monitor how well the
minion output matches the profile over time.